In [1]:
!pip install -q --upgrade diffusers
# Then restart the kernel and rerun from the pipe loading cell

In [ ]:
"""
Stable Video Diffusion (SVD) – Image-to-Video Generation
=========================================================
Uses Stability AI's stable-video-diffusion-img2vid-xt model to animate
a static image into a short video clip. Optimized for Kaggle's T4 GPU
(16 GB VRAM) using CPU offloading and chunked attention/decoding.
"""

import os
import torch
from diffusers import StableVideoDiffusionPipeline
from diffusers.utils import load_image, export_to_video
from IPython.display import Video, display
from PIL import Image

# ---------------------------------------------------------------------------
# GPU Memory Setup
# ---------------------------------------------------------------------------
# Free any cached VRAM from previous runs before allocating new tensors.
torch.cuda.empty_cache()

# Tell PyTorch's CUDA allocator to use expandable memory segments instead of
# fixed-size blocks. This reduces "out of memory" errors caused by
# fragmentation, particularly useful when tensor sizes vary across steps.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---------------------------------------------------------------------------
# Model Loading
# ---------------------------------------------------------------------------
# SVD XT is ~7 GB on disk. We load it in float16 (half precision) to halve
# VRAM usage compared to the default float32, with negligible quality loss.
pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=torch.float16,   # use 16-bit floats to halve VRAM usage
    variant="fp16",              # load the pre-quantized fp16 weight files
)

# Automatically move model layers between CPU and GPU as needed.
# Layers not currently in use are offloaded to CPU RAM, freeing VRAM for
# the active computation. Essential for fitting SVD on a single T4.
pipe.enable_model_cpu_offload()

# Process the UNet's attention in smaller sequential chunks rather than
# computing the full attention matrix at once. Trades a small speed cost
# for significantly lower peak VRAM usage during the diffusion steps.
pipe.unet.enable_forward_chunking()

# ---------------------------------------------------------------------------
# Input Image
# ---------------------------------------------------------------------------
# SVD XT expects exactly 1024×576 pixels (16:9 widescreen aspect ratio).
# Resizing to any other resolution will cause an error or degraded output.
image = load_image(
    "https://huggingface.co/datasets/huggingface/documentation-images"
    "/resolve/main/diffusers/guitar-man.png"
).resize((1024, 576))

# ---------------------------------------------------------------------------
# Video Generation
# ---------------------------------------------------------------------------
frames = pipe(
    image,
    num_frames=25,              # total frames to generate (~3.5 s at 7 fps)
    num_inference_steps=25,     # denoising steps; more = higher quality but slower
    decode_chunk_size=4,        # decode 4 latent frames at a time to save VRAM
    motion_bucket_id=24,        # controls motion intensity: 1 (subtle) – 255 (chaotic)
    noise_aug_strength=0.02,    # slight noise on the conditioning image; helps
                                # the model generalise motion rather than copy pixels
    generator=torch.Generator("cpu").manual_seed(666),  # fixed seed for reproducibility
).frames[0]                     # [0] unwraps the batch dimension (batch size = 1)

# ---------------------------------------------------------------------------
# Export & Display
# ---------------------------------------------------------------------------
output_path = "/kaggle/working/output.mp4"

export_to_video(frames, output_path, fps=7)
print(f"Video saved to: {output_path}")

# Render the video inline inside the Kaggle / Jupyter notebook.
display(Video(output_path, embed=True, width=640))

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/520 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

Done!
